# LeetCode 1685 — Sum of Absolute Differences in a Sorted Array bằng Python

Link đề: https://leetcode.com/problems/sum-of-absolute-differences-in-a-sorted-array/ · Độ khó: **Medium** · Chủ đề: Array, Math, Prefix Sum

Skeleton LeetCode:

```python
class Solution:
    def getSumAbsoluteDifferences(self, nums: List[int]) -> List[int]:
        ...
```

Notebook đi từ **tính trực tiếp mọi cặp** `O(n²)` → **tổng tiền tố** tách trị tuyệt đối thành hai phía → **hai biến tổng chạy** `O(1)` bộ nhớ phụ → **cập nhật gia tăng** `res[i]` từ `res[i−1]` với một công thức một dòng.

## 1. Mục tiêu học

1. Biết **phá dấu trị tuyệt đối** nhờ mảng đã sắp xếp: bên trái `nums[i] − nums[j] ≥ 0`, bên phải `nums[j] − nums[i] ≥ 0`.
2. Biến tổng `Σ |nums[i] − nums[j]|` thành công thức theo **tổng tiền tố**: `x·i − pre[i] + (suf − x·(n − 1 − i))`.
3. Cài đặt bằng mảng tiền tố, bằng hai biến tổng chạy, và bằng **sai phân** `res[i] − res[i−1] = d·(2i − n)`.
4. Nhận diện các lỗi lệch một (đếm phần tử bên phải, có/không gồm chính `i`) và lỗi dùng công thức trên mảng **chưa sắp xếp**.
5. Kiểm thử đối chiếu với `O(n²)` trên dữ liệu ngẫu nhiên đã sắp xếp, seed cố định.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums` đã **sắp xếp không giảm**. Hãy dựng và trả về mảng `result` cùng độ dài, trong đó `result[i]` là **tổng các hiệu tuyệt đối** giữa `nums[i]` và **mọi phần tử khác**:

`result[i] = Σ_{j ≠ i} |nums[i] − nums[j]|`

**Ví dụ 1:** `nums = [2,3,5]` → `[4,3,5]`.
- `result[0] = |2−2| + |2−3| + |2−5| = 0 + 1 + 3 = 4`
- `result[1] = |3−2| + |3−3| + |3−5| = 1 + 0 + 2 = 3`
- `result[2] = |5−2| + |5−3| + |5−5| = 3 + 2 + 0 = 5`

**Ví dụ 2:** `nums = [1,4,6,8,10]` → `[24,15,13,15,21]`.

**Ràng buộc:**
- `2 ≤ nums.length ≤ 10⁵`
- `1 ≤ nums[i] ≤ nums[i+1] ≤ 10⁴`

In [1]:
import bisect
import random
import time
from itertools import accumulate
from typing import List

examples = [([2, 3, 5], [4, 3, 5]), ([1, 4, 6, 8, 10], [24, 15, 13, 15, 21])]

for nums, exp in examples:
    for i, x in enumerate(nums):
        terms = [abs(x - y) for y in nums]
        print(f"nums={nums}, i={i}: {' + '.join(map(str, terms))} = {sum(terms)}")
    print("kỳ vọng:", exp)

nums=[2, 3, 5], i=0: 0 + 1 + 3 = 4
nums=[2, 3, 5], i=1: 1 + 0 + 2 = 3
nums=[2, 3, 5], i=2: 3 + 2 + 0 = 5
kỳ vọng: [4, 3, 5]
nums=[1, 4, 6, 8, 10], i=0: 0 + 3 + 5 + 7 + 9 = 24
nums=[1, 4, 6, 8, 10], i=1: 3 + 0 + 2 + 4 + 6 = 15
nums=[1, 4, 6, 8, 10], i=2: 5 + 2 + 0 + 2 + 4 = 13
nums=[1, 4, 6, 8, 10], i=3: 7 + 4 + 2 + 0 + 2 = 15
nums=[1, 4, 6, 8, 10], i=4: 9 + 6 + 4 + 2 + 0 = 21
kỳ vọng: [24, 15, 13, 15, 21]


## 3. Kiến thức nền

### 3.1 Trị tuyệt đối trên mảng đã sắp xếp

Với `nums` không giảm và vị trí `i` cố định:
- `j < i` ⇒ `nums[j] ≤ nums[i]` ⇒ `|nums[i] − nums[j]| = nums[i] − nums[j]`.
- `j > i` ⇒ `nums[j] ≥ nums[i]` ⇒ `|nums[i] − nums[j]| = nums[j] − nums[i]`.

In [2]:
nums = [1, 4, 6, 8, 10]
i, x = 2, nums[2]
print("bên trái :", [x - y for y in nums[:i]], "(đều >= 0)")
print("bên phải :", [y - x for y in nums[i + 1:]], "(đều >= 0)")

bên trái : [5, 2] (đều >= 0)
bên phải : [2, 4] (đều >= 0)


### 3.2 Tổng tiền tố

`pre[k] = nums[0] + … + nums[k−1]` (`pre[0] = 0`). Tổng `nums[a..b−1]` = `pre[b] − pre[a]`. `itertools.accumulate(nums, initial=0)` dựng `pre`.

In [3]:
pre = list(accumulate(nums, initial=0))
print("pre =", pre)
print("tổng bên trái của i=2:", pre[2], "| tổng bên phải:", pre[5] - pre[3])

pre = [0, 1, 5, 11, 19, 29]
tổng bên trái của i=2: 5 | tổng bên phải: 18


### 3.3 Tổng hai biến chạy

Không cần cả mảng `pre`: duy trì `left = tổng các phần tử đã đi qua` và `total = tổng cả mảng`; tổng bên phải = `total − left − x`.

In [4]:
total, left = sum(nums), 0
for i, x in enumerate(nums):
    print(f"i={i}: left={left}, right={total - left - x}")
    left += x

i=0: left=0, right=28
i=1: left=1, right=24
i=2: left=5, right=18
i=3: left=11, right=10
i=4: left=19, right=0


### 3.4 Sai phân của một dãy

Nếu biết `res[0]` và hiệu `res[i] − res[i−1]` thì `res[i] = res[0] + Σ` các hiệu. Đôi khi hiệu có công thức gọn hơn chính `res[i]`.

In [5]:
seq = [24, 15, 13, 15, 21]
print("hiệu liên tiếp:", [b - a for a, b in zip(seq, seq[1:])])

hiệu liên tiếp: [-9, -2, 2, 6]


## 4. Ý tưởng nền tảng / trực giác

Tách tổng tại vị trí `i` (với `x = nums[i]`, `n = len(nums)`):

`result[i] = Σ_{j<i} (x − nums[j]) + Σ_{j>i} (nums[j] − x)`
`          = x·i − (tổng bên trái) + (tổng bên phải) − x·(n − 1 − i)`

Tính tay ví dụ 2 tại `i = 2` (`x = 6`): bên trái `1 + 4 = 5` (2 phần tử), bên phải `8 + 10 = 18` (2 phần tử):
`6·2 − 5 + 18 − 6·2 = 12 − 5 + 18 − 12 = 13` ✔.

Hình dung trên trục số: `result[i]` là **tổng khoảng cách** từ điểm `x` tới mọi điểm khác. Các điểm bên trái "kéo" `x` sang trái, các điểm bên phải kéo sang phải; tổng khoảng cách nhỏ nhất ở **trung vị** (ví dụ 2: nhỏ nhất `13` tại phần tử giữa).

In [6]:
nums = [1, 4, 6, 8, 10]
n = len(nums)
for i, x in enumerate(nums):
    L, R = sum(nums[:i]), sum(nums[i + 1:])
    print(f"i={i}, x={x:>2}: x*i - L + R - x*(n-1-i) = {x}*{i} - {L} + {R} - {x}*{n - 1 - i} = {x * i - L + R - x * (n - 1 - i)}")

i=0, x= 1: x*i - L + R - x*(n-1-i) = 1*0 - 0 + 28 - 1*4 = 24
i=1, x= 4: x*i - L + R - x*(n-1-i) = 4*1 - 1 + 24 - 4*3 = 15
i=2, x= 6: x*i - L + R - x*(n-1-i) = 6*2 - 5 + 18 - 6*2 = 13
i=3, x= 8: x*i - L + R - x*(n-1-i) = 8*3 - 11 + 10 - 8*1 = 15
i=4, x=10: x*i - L + R - x*(n-1-i) = 10*4 - 19 + 0 - 10*0 = 21


Vẽ các điểm trên trục số và "đo" tổng khoảng cách từ mỗi điểm. Dãy `result` **không tăng ở nửa đầu, không giảm ở nửa sau** (dấu của hiệu là dấu của `2i − n`, mục 6), đáy ở trung vị — ta kiểm chứng trên nhiều mảng ngẫu nhiên:

In [7]:
nums = [1, 4, 6, 8, 10]
line = [' '] * 11
for x in nums:
    line[x] = '●'
print("trục số 0..10:", ''.join(line))
res = [sum(abs(x - y) for y in nums) for x in nums]
for x, r in zip(nums, res):
    print(f"  từ {x:>2}: tổng khoảng cách = {r:>2} " + '█' * r)

rnd = random.Random(7)
for _ in range(500):
    a = sorted(rnd.randint(1, 30) for _ in range(rnd.randint(2, 12)))
    r = [sum(abs(x - y) for y in a) for x in a]
    med = a[(len(a) - 1) // 2]
    assert min(r) == sum(abs(med - y) for y in a)                       # cực tiểu tại trung vị
    n_ = len(a)
    assert all(r[i] <= r[i - 1] for i in range(1, n_) if 2 * i < n_)     # nửa đầu: không tăng
    assert all(r[i] >= r[i - 1] for i in range(1, n_) if 2 * i > n_)     # nửa sau: không giảm
print("500 mảng ngẫu nhiên: cực tiểu ở trung vị, result không tăng ở nửa đầu và không giảm ở nửa sau: OK")

trục số 0..10:  ●  ● ● ● ●
  từ  1: tổng khoảng cách = 24 ████████████████████████
  từ  4: tổng khoảng cách = 15 ███████████████
  từ  6: tổng khoảng cách = 13 █████████████
  từ  8: tổng khoảng cách = 15 ███████████████
  từ 10: tổng khoảng cách = 21 █████████████████████
500 mảng ngẫu nhiên: cực tiểu ở trung vị, result không tăng ở nửa đầu và không giảm ở nửa sau: OK


## 5. Cách 1 — Brute Force / baseline: hai vòng lặp

In [8]:
def sum_abs_brute(nums):
    n = len(nums)
    res = []
    for i in range(n):
        s = 0
        for j in range(n):
            s += abs(nums[i] - nums[j])          # j == i đóng góp 0, không cần loại
        res.append(s)
    return res

for nums_, exp in examples:
    print(sum_abs_brute(nums_), "(kỳ vọng", exp, ")")

[4, 3, 5] (kỳ vọng [4, 3, 5] )
[24, 15, 13, 15, 21] (kỳ vọng [24, 15, 13, 15, 21] )


**Độ phức tạp:** `O(n²)` thời gian — với `n = 10⁵` là `10¹⁰` phép tính: quá chậm. Bộ nhớ `O(1)` phụ (ngoài kết quả).

**Nút thắt:** ta không dùng tới tính **đã sắp xếp**. Nhờ nó, mọi trị tuyệt đối bên trái/phải đều có **cùng dấu**, nên tổng của chúng gom được thành **tổng của một đoạn** — tính `O(1)` với tổng tiền tố.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** với mảng không giảm,

`result[i] = (x·i − pre[i]) + ((pre[n] − pre[i+1]) − x·(n − 1 − i))`

trong đó `pre[i]` là tổng `i` phần tử đầu (bên trái `i`), `pre[n] − pre[i+1]` là tổng các phần tử bên phải `i`.

**Quy tắc "nếu… thì…":** **nếu** mảng đã sắp xếp **thì** phần trái đóng góp `x·(số phần tử trái) − (tổng trái)` và phần phải đóng góp `(tổng phải) − x·(số phần tử phải)`.

**Sai phân:** đi từ điểm `nums[i−1]` sang điểm `nums[i]`, ta dịch sang phải một đoạn `d = nums[i] − nums[i−1] ≥ 0`. Xét khoảng cách tới từng phần tử `nums[j]`:
- `j ≤ i − 1` (có `i` phần tử, đều `≤ nums[i−1]`): khoảng cách **tăng** đúng `d`;
- `j ≥ i` (có `n − i` phần tử, đều `≥ nums[i]`): khoảng cách **giảm** đúng `d`.

(Kể cả `j = i − 1`: từ `0` thành `d`; và `j = i`: từ `d` thành `0`.) Vậy

`result[i] = result[i−1] + d·i − d·(n − i) = result[i−1] + d·(2i − n)`.

**Bất biến (hai biến chạy):** trước khi xử lý `i`, `left = Σ_{j<i} nums[j]`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Mảng tổng tiền tố

In [9]:
def sum_abs_prefix(nums):
    n = len(nums)
    pre = list(accumulate(nums, initial=0))
    res = []
    for i, x in enumerate(nums):
        left_sum = pre[i]
        right_sum = pre[n] - pre[i + 1]
        res.append(x * i - left_sum + right_sum - x * (n - 1 - i))
    return res

for nums_, exp in examples:
    print(sum_abs_prefix(nums_), "(kỳ vọng", exp, ")")

[4, 3, 5] (kỳ vọng [4, 3, 5] )
[24, 15, 13, 15, 21] (kỳ vọng [24, 15, 13, 15, 21] )


**Độ phức tạp:** `O(n)` thời gian, `O(n)` bộ nhớ cho `pre`.

### 7.2 Cách 3 — Hai biến tổng chạy (lời giải chính)

In [10]:
def sum_abs_running(nums):
    n = len(nums)
    total = sum(nums)
    left = 0
    res = []
    for i, x in enumerate(nums):
        right = total - left - x
        res.append(x * i - left + right - x * (n - 1 - i))
        left += x                       # cập nhật SAU khi dùng
    return res

for nums_, exp in examples:
    print(sum_abs_running(nums_), "(kỳ vọng", exp, ")")

[4, 3, 5] (kỳ vọng [4, 3, 5] )
[24, 15, 13, 15, 21] (kỳ vọng [24, 15, 13, 15, 21] )


Có thể rút gọn: `x·i − left + (total − left − x) − x·(n − 1 − i) = total − 2·left + x·(2i − n)`. **Độ phức tạp:** `O(n)` thời gian, `O(1)` bộ nhớ phụ.

In [11]:
def sum_abs_compact(nums):
    n, total, left = len(nums), sum(nums), 0
    res = []
    for i, x in enumerate(nums):
        res.append(total - 2 * left + x * (2 * i - n))
        left += x
    return res

for nums_, exp in examples:
    print(sum_abs_compact(nums_), "(kỳ vọng", exp, ")")

[4, 3, 5] (kỳ vọng [4, 3, 5] )
[24, 15, 13, 15, 21] (kỳ vọng [24, 15, 13, 15, 21] )


### 7.3 Cách 4 — Cập nhật gia tăng bằng sai phân

`result[0] = Σ (nums[j] − nums[0]) = total − n·nums[0]`; sau đó `result[i] = result[i−1] + (nums[i] − nums[i−1])·(2i − n)`.

In [12]:
def sum_abs_delta(nums):
    n = len(nums)
    res = [sum(nums) - n * nums[0]]
    for i in range(1, n):
        d = nums[i] - nums[i - 1]
        res.append(res[-1] + d * (2 * i - n))
    return res

for nums_, exp in examples:
    print(sum_abs_delta(nums_), "(kỳ vọng", exp, ")")

[4, 3, 5] (kỳ vọng [4, 3, 5] )
[24, 15, 13, 15, 21] (kỳ vọng [24, 15, 13, 15, 21] )


**Độ phức tạp:** `O(n)` thời gian, `O(1)` bộ nhớ phụ. Công thức cho thấy rõ: `result` **giảm** khi `2i < n` (nửa đầu) và **tăng** khi `2i > n` (nửa sau) — cực tiểu ở trung vị.

## 8. TRACE từng bước

Trace cách 3 trên ví dụ 2: `nums = [1,4,6,8,10]`, `total = 29`, `n = 5`.

| i | x | left | right = total − left − x | x·i − left | right − x·(n−1−i) | result |
|---|---|---|---|---|---|---|
| 0 | 1 | 0 | 28 | 0 | 28 − 4 = 24 | 24 |
| 1 | 4 | 1 | 24 | 3 | 24 − 12 = 12 | 15 |
| 2 | 6 | 5 | 18 | 7 | 18 − 12 = 6 | 13 |
| 3 | 8 | 11 | 10 | 13 | 10 − 8 = 2 | 15 |
| 4 | 10 | 19 | 0 | 21 | 0 | 21 |

In [13]:
def trace_running(nums):
    n, total, left = len(nums), sum(nums), 0
    print(f"nums={nums}, n={n}, total={total}")
    res = []
    for i, x in enumerate(nums):
        right = total - left - x
        a, b = x * i - left, right - x * (n - 1 - i)
        res.append(a + b)
        print(f"  i={i} x={x:>2}: left={left:>2}, right={right:>2} | trái {x}*{i}-{left}={a:>2} | phải {right}-{x}*{n - 1 - i}={b:>2} | result={a + b}")
        left += x
    return res

trace_running([1, 4, 6, 8, 10])

nums=[1, 4, 6, 8, 10], n=5, total=29
  i=0 x= 1: left= 0, right=28 | trái 1*0-0= 0 | phải 28-1*4=24 | result=24
  i=1 x= 4: left= 1, right=24 | trái 4*1-1= 3 | phải 24-4*3=12 | result=15
  i=2 x= 6: left= 5, right=18 | trái 6*2-5= 7 | phải 18-6*2= 6 | result=13
  i=3 x= 8: left=11, right=10 | trái 8*3-11=13 | phải 10-8*1= 2 | result=15
  i=4 x=10: left=19, right= 0 | trái 10*4-19=21 | phải 0-10*0= 0 | result=21


[24, 15, 13, 15, 21]

Trace cách 4 (sai phân) trên cùng ví dụ và trên mảng có phần tử trùng `[2, 2, 3, 7, 7, 7]` (ví dụ tự dựng — kiểm chứng bằng brute force bên dưới):

In [14]:
def trace_delta(nums):
    n = len(nums)
    r = sum(nums) - n * nums[0]
    print(f"  result[0] = total - n*nums[0] = {sum(nums)} - {n}*{nums[0]} = {r}")
    out = [r]
    for i in range(1, n):
        d = nums[i] - nums[i - 1]
        r += d * (2 * i - n)
        out.append(r)
        print(f"  i={i}: d={d}, 2i-n={2 * i - n:>2}, result[{i}] = {r}")
    return out

print("[1,4,6,8,10]:"); trace_delta([1, 4, 6, 8, 10])
print("[2,2,3,7,7,7]:"); r = trace_delta([2, 2, 3, 7, 7, 7]); print("  brute:", sum_abs_brute([2, 2, 3, 7, 7, 7]), "| khớp:", r == sum_abs_brute([2, 2, 3, 7, 7, 7]))

[1,4,6,8,10]:
  result[0] = total - n*nums[0] = 29 - 5*1 = 24
  i=1: d=3, 2i-n=-3, result[1] = 15
  i=2: d=2, 2i-n=-1, result[2] = 13
  i=3: d=2, 2i-n= 1, result[3] = 15
  i=4: d=2, 2i-n= 3, result[4] = 21
[2,2,3,7,7,7]:
  result[0] = total - n*nums[0] = 28 - 6*2 = 16
  i=1: d=0, 2i-n=-4, result[1] = 16
  i=2: d=1, 2i-n=-2, result[2] = 14
  i=3: d=4, 2i-n= 0, result[3] = 14
  i=4: d=0, 2i-n= 2, result[4] = 14
  i=5: d=0, 2i-n= 4, result[5] = 14
  brute: [16, 16, 14, 14, 14, 14] | khớp: True


## 9. Code nộp LeetCode

In [15]:
class Solution:
    def getSumAbsoluteDifferences(self, nums: List[int]) -> List[int]:
        n = len(nums)
        total = sum(nums)
        left = 0                              # tổng các phần tử bên trái i
        res = []
        for i, x in enumerate(nums):
            right = total - left - x          # tổng các phần tử bên phải i
            res.append(x * i - left + right - x * (n - 1 - i))
            left += x
        return res

for nums_, exp in examples:
    assert Solution().getSumAbsoluteDifferences(nums_) == exp
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

Cố định `i`, `x = nums[i]`. Vì `nums` không giảm:
- với `j < i`: `nums[j] ≤ x` ⇒ `|x − nums[j]| = x − nums[j]`; tổng trên `i` chỉ số này là `x·i − Σ_{j<i} nums[j]`;
- với `j > i`: `nums[j] ≥ x` ⇒ `|x − nums[j]| = nums[j] − x`; tổng trên `n − 1 − i` chỉ số là `Σ_{j>i} nums[j] − x·(n − 1 − i)`;
- `j = i` đóng góp 0.

Cộng lại được đúng công thức. Trong cách 3, bất biến `left = Σ_{j<i} nums[j]` đúng trước mỗi bước (khởi tạo `0`, cộng `x` **sau** khi dùng), và `right = total − left − x = Σ_{j>i} nums[j]`.

Cách 4 đúng vì `result[i] − result[i−1] = Σ_j (|nums[i] − nums[j]| − |nums[i−1] − nums[j]|)`; với `j ≤ i − 1` mỗi hạng tử là `+d`, với `j ≥ i` là `−d` (nhờ sắp xếp), nên hiệu `= d·i − d·(n − i) = d·(2i − n)`. Và `result[0] = Σ_j (nums[j] − nums[0])`.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space phụ |
|---|---|---|
| 1. Hai vòng lặp | `O(n²)` | `O(1)` |
| 2. Mảng tiền tố | `O(n)` | `O(n)` |
| 3. Hai biến chạy | `O(n)` | `O(1)` |
| 4. Sai phân | `O(n)` | `O(1)` |

(Không tính mảng kết quả `O(n)` bắt buộc phải trả về.) Giá trị lớn nhất `≈ n · 10⁴ = 10⁹` — vừa với `int` 32-bit có dấu? `10⁵ · 10⁴ = 10⁹ < 2.147·10⁹` nên vừa; trong Python thì không lo tràn.

In [16]:
rnd = random.Random(1685)
big = sorted(rnd.randint(1, 10**4) for _ in range(10**5))
for name, fn in [("prefix", sum_abs_prefix), ("running", sum_abs_running), ("compact", sum_abs_compact), ("delta", sum_abs_delta), ("Solution", Solution().getSumAbsoluteDifferences)]:
    t0 = time.perf_counter(); r = fn(big); dt = time.perf_counter() - t0
    print(f"{name:<9}: max={max(r):>10}, {dt * 1000:6.1f} ms")
small = big[::50][:2000]
t0 = time.perf_counter(); r = sum_abs_brute(small); dt = time.perf_counter() - t0
print(f"brute n={len(small)}: {dt * 1000:.0f} ms | khớp running: {r == sum_abs_running(small)}")

prefix   : max= 500805786,   19.7 ms
running  : max= 500805786,   15.6 ms
compact  : max= 500805786,   12.1 ms
delta    : max= 500805786,   11.3 ms
Solution : max= 500805786,   15.3 ms


brute n=2000: 172 ms | khớp running: True


## 12. Bộ test

In [17]:
ALL = {
    "prefix": sum_abs_prefix,
    "running": sum_abs_running,
    "compact": sum_abs_compact,
    "delta": sum_abs_delta,
    "Solution": lambda a: Solution().getSumAbsoluteDifferences(a),
}

def check(nums, exp=None, brute=True):
    ref = sum_abs_brute(nums) if brute else sum_abs_running(nums)
    for name, fn in ALL.items():
        got = fn(list(nums))
        assert got == ref, (name, nums, got, ref)
    if exp is not None:
        assert ref == exp, (nums, ref, exp)
    return ref

for nums_, exp in examples:
    check(nums_, exp)
print("Ví dụ đề: OK")

check([1, 1], [0, 0])
check([5, 5, 5, 5], [0, 0, 0, 0])
check([1, 10000], [9999, 9999])
check([1, 2], [1, 1])
check([2, 2, 3, 7, 7, 7])
print("Edge case: OK")

rnd = random.Random(2011)
for _ in range(2000):
    n = rnd.randint(2, 30)
    nums_ = sorted(rnd.randint(1, rnd.choice([3, 20, 10**4])) for _ in range(n))
    check(nums_)
print("2000 mảng ngẫu nhiên đã sắp xếp so với brute force: OK")

for _ in range(10):
    n = rnd.randint(1000, 10**5)
    nums_ = sorted(rnd.randint(1, 10**4) for _ in range(n))
    a = Solution().getSumAbsoluteDifferences(nums_)
    assert a == sum_abs_prefix(nums_) == sum_abs_delta(nums_)
print("10 mảng cỡ lớn: các cách nhanh khớp nhau: OK")

Ví dụ đề: OK
Edge case: OK
2000 mảng ngẫu nhiên đã sắp xếp so với brute force: OK


10 mảng cỡ lớn: các cách nhanh khớp nhau: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Đếm sai số phần tử bên phải (gồm cả `i`) nhưng tổng lại không gồm `x`

In [18]:
def sum_abs_bug_count(nums):
    n = len(nums)
    pre = list(accumulate(nums, initial=0))
    return [x * i - pre[i] + (pre[n] - pre[i + 1]) - x * (n - i) for i, x in enumerate(nums)]   # SAI: n - i

nums_ = [2, 3, 5]
print("sai :", sum_abs_bug_count(nums_), "| đúng:", Solution().getSumAbsoluteDifferences(nums_))

sai : [2, 0, 0] | đúng: [4, 3, 5]


**Vì sao sai:** tổng bên phải `pre[n] − pre[i+1]` **không** gồm `x`, nhưng số lượng `n − i` lại **gồm** `x` ⇒ trừ thừa đúng một `x`. **Sửa:** dùng `n − 1 − i` (hoặc dùng `pre[n] − pre[i]` với `n − i` — gồm cả hai, và `x − x = 0`).

### Lỗi 2 — Cập nhật `left` **trước** khi dùng

In [19]:
def sum_abs_bug_update_order(nums):
    n, total, left = len(nums), sum(nums), 0
    res = []
    for i, x in enumerate(nums):
        left += x                                        # SAI: left giờ gồm cả x
        right = total - left - x
        res.append(x * i - left + right - x * (n - 1 - i))
    return res

print("sai :", sum_abs_bug_update_order([2, 3, 5]), "| đúng:", Solution().getSumAbsoluteDifferences([2, 3, 5]))

sai : [0, -3, -5] | đúng: [4, 3, 5]


**Vì sao sai:** `left` phải là tổng **bên trái** `i` (không gồm `x`). Cộng trước làm cả `left` và `right` lệch. **Sửa:** `left += x` ở **cuối** vòng lặp.

### Lỗi 3 — Sai dấu trong công thức sai phân

In [20]:
def sum_abs_bug_delta_sign(nums):
    n = len(nums)
    res = [sum(nums) - n * nums[0]]
    for i in range(1, n):
        res.append(res[-1] + (nums[i] - nums[i - 1]) * (n - 2 * i))   # SAI: đảo dấu
    return res

print("sai :", sum_abs_bug_delta_sign([1, 4, 6, 8, 10]), "| đúng:", Solution().getSumAbsoluteDifferences([1, 4, 6, 8, 10]))

sai : [24, 33, 35, 33, 27] | đúng: [24, 15, 13, 15, 21]


**Vì sao sai:** dịch sang phải `d`, khoảng cách tới **`i` phần tử bên trái** tăng, tới **`n − i` phần tử bên phải** giảm ⇒ `+d·i − d·(n − i) = d·(2i − n)`. **Sửa:** `(2 * i - n)`. Mẹo kiểm: phần tử đầu tiên (`i = 1`) gần như luôn làm tổng **giảm** khi `n > 2`.

### Lỗi 4 — Áp dụng công thức trên mảng **chưa sắp xếp**

In [21]:
unsorted = [5, 2, 3]
print("công thức trên mảng chưa sắp:", Solution().getSumAbsoluteDifferences(unsorted), "| đúng (brute):", sum_abs_brute(unsorted))

công thức trên mảng chưa sắp: [-5, -2, -1] | đúng (brute): [5, 4, 3]


**Vì sao sai:** việc "phá dấu" trị tuyệt đối dựa trên `nums[j] ≤ nums[i]` với `j < i`. Đề bảo đảm mảng đã sắp nên lời giải đúng; nhưng khi dùng lại cho bài khác hãy nhớ điều kiện này. **Sửa (biến thể):** sắp kèm chỉ số rồi tính, hoặc dùng `bisect` trên bản đã sắp (mục 15).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Tổng `|a_i − a_j|`" + mảng **đã sắp xếp** ⇒ phá dấu: trái là `x − a_j`, phải là `a_j − x` ⇒ **tổng tiền tố**.
- Công thức gọn: `res[i] = total − 2·left + x·(2i − n)`.
- Cập nhật biến chạy **sau** khi dùng để giữ bất biến "chỉ gồm phần bên trái".
- Kiểm nhanh bằng ví dụ nhỏ có phần tử trùng (`[5,5,5]` ⇒ toàn 0) và hai phần tử (`[1, 10]` ⇒ `[9, 9]`).
- Tổng khoảng cách nhỏ nhất đạt tại **trung vị** — kiến thức hay dùng trong các bài "dời các điểm về một chỗ".

## 15. Biến thể của bài

1. **Mảng chưa sắp xếp:** sắp kèm chỉ số gốc, tính trên bản đã sắp, rồi đặt kết quả về đúng vị trí — `O(n log n)`.
2. **Truy vấn với điểm `q` bất kỳ** (không thuộc mảng): `k = bisect_left(a, q)`, tổng = `q·k − pre[k] + (pre[n] − pre[k]) − q·(n − k)`.
3. **LeetCode 2602 — Minimum Operations to Make All Array Elements Equal:** chính là biến thể 2 cho nhiều truy vấn.
4. **LeetCode 462 — Minimum Moves to Equal Array Elements II:** điểm tối ưu là trung vị.
5. **LeetCode 2121 — Intervals Between Identical Elements:** cùng kỹ thuật trên danh sách chỉ số của mỗi giá trị.
6. **Tổng `|a_i − a_j|` trên mọi cặp:** `Σ_i a_i·(2i − n + 1)` trên mảng đã sắp (mỗi cặp tính một lần).

In [22]:
def sum_abs_unsorted(nums):
    order = sorted(range(len(nums)), key=lambda i: nums[i])
    sorted_vals = [nums[i] for i in order]
    res_sorted = Solution().getSumAbsoluteDifferences(sorted_vals)
    res = [0] * len(nums)
    for pos, i in enumerate(order):
        res[i] = res_sorted[pos]
    return res

def query_sum_abs(a_sorted, pre, q):
    n = len(a_sorted)
    k = bisect.bisect_left(a_sorted, q)
    return q * k - pre[k] + (pre[n] - pre[k]) - q * (n - k)

def all_pairs_sum(a_sorted):
    n = len(a_sorted)
    return sum(x * (2 * i - n + 1) for i, x in enumerate(a_sorted))

print("chưa sắp [5,2,3]:", sum_abs_unsorted([5, 2, 3]), "| brute:", sum_abs_brute([5, 2, 3]))
a = [1, 4, 6, 8, 10]; pre = list(accumulate(a, initial=0))
print("truy vấn q=5:", query_sum_abs(a, pre, 5), "| kiểm:", sum(abs(5 - x) for x in a))
print("tổng mọi cặp:", all_pairs_sum(a), "| kiểm:", sum(abs(x - y) for i, x in enumerate(a) for y in a[i + 1:]), "| = tổng kết quả / 2:", sum(Solution().getSumAbsoluteDifferences(a)) // 2)

chưa sắp [5,2,3]: [5, 4, 3] | brute: [5, 4, 3]
truy vấn q=5: 14 | kiểm: 14
tổng mọi cặp: 44 | kiểm: 44 | = tổng kết quả / 2: 44


Cài đặt biến thể 5 (LeetCode 2121 — Intervals Between Identical Elements): với mỗi giá trị, lấy danh sách **chỉ số** của nó (tự nhiên đã tăng dần) và áp dụng đúng công thức của bài này trên danh sách chỉ số.

In [23]:
from collections import defaultdict

def get_distances(arr):
    pos = defaultdict(list)
    for i, v in enumerate(arr):
        pos[v].append(i)
    res = [0] * len(arr)
    for v, idxs in pos.items():
        sums = Solution().getSumAbsoluteDifferences(idxs) if len(idxs) >= 2 else [0]
        for i, s_ in zip(idxs, sums):
            res[i] = s_
    return res

arr = [2, 1, 3, 1, 2, 3, 3]
print("2121:", get_distances(arr), "| kiểm:", [sum(abs(i - j) for j in range(len(arr)) if arr[j] == arr[i]) for i in range(len(arr))])

2121: [4, 2, 7, 2, 4, 4, 5] | kiểm: [4, 2, 7, 2, 4, 4, 5]


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space phụ | Khi nào dùng |
|---|---|---|---|---|
| 1. Hai vòng lặp | tính thẳng định nghĩa | `O(n²)` | `O(1)` | đối chiếu |
| 2. Mảng tiền tố | phá dấu + `pre` | `O(n)` | `O(n)` | rõ ràng, dễ mở rộng truy vấn |
| 3. Hai biến chạy | `left`, `total` | `O(n)` | `O(1)` | **nộp bài** |
| 4. Sai phân | `res[i] = res[i−1] + d(2i − n)` | `O(n)` | `O(1)` | đẹp, thấy rõ cực tiểu ở trung vị |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `Solution().getSumAbsoluteDifferences([1, 1, 4])` và `([3, 3, 3])`?

**Bài 2 (tính tay):** Dùng công thức sai phân, tính `result` cho `nums = [1, 2, 4, 7, 11]` (ghi `d` và `2i − n` từng bước).

**Bài 3 (điền TODO):** Hoàn thiện `min_total_distance(nums)` trả về `min(result)` và **một** chỉ số đạt min, chỉ một lượt duyệt, không lưu cả mảng `result`.

**Bài 4 (tự viết lại):** Viết `sum_abs_queries(nums, queries)` (mảng `nums` **chưa** sắp) trả về, với mỗi `q`, tổng `Σ |nums[j] − q|`, trong `O((n + m) log n)`.

**Bài 5 (mini challenge):** Viết `sum_sq_diff(nums)` trả về mảng `Σ_j (nums[i] − nums[j])²` cho mọi `i` trong `O(n)` (không cần sắp xếp!). Gợi ý: khai triển bình phương.

In [24]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def min_total_distance(nums):
    # TODO: như sum_abs_compact nhưng giữ (best, best_i) thay vì list
    pass

# Bài 4 — TODO
def sum_abs_queries(nums, queries):
    # TODO: a = sorted(nums); pre = tổng tiền tố; mỗi q dùng bisect_left
    pass

# Bài 5 — TODO
def sum_sq_diff(nums):
    # TODO: (x - y)^2 = x^2 - 2xy + y^2 => n*x^2 - 2x*S1 + S2
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1**
- Mức 1: Với `[1, 1, 4]`: hai số 1 cách 4 một khoảng 3.
- Mức 2: `result = [3, 3, 6]`.
- Mức 3: Mảng toàn số bằng nhau ⇒ toàn 0.

**Bài 2**
- Mức 1: `result[0] = total − n·nums[0] = 25 − 5 = 20`.
- Mức 2: `d` lần lượt `1, 2, 3, 4`; `2i − n` lần lượt `−3, −1, 1, 3`.
- Mức 3: `20 → 17 → 15 → 18 → 30`.

**Bài 3**
- Mức 1: Chỉ cần so sánh từng giá trị khi tính xong.
- Mức 2: Dùng `total − 2·left + x·(2i − n)`.
- Mức 3: Chỉ số đạt min là một trung vị.

**Bài 4**
- Mức 1: Sắp xếp một lần, tiền tố một lần.
- Mức 2: `k = bisect_left(a, q)` = số phần tử `< q`.
- Mức 3: Tổng = `q·k − pre[k] + (pre[n] − pre[k]) − q·(n − k)`.

**Bài 5**
- Mức 1: `S1 = Σ y`, `S2 = Σ y²`.
- Mức 2: `Σ_j (x − y_j)² = n·x² − 2x·S1 + S2`.
- Mức 3: Hạng tử `j = i` bằng 0 nên không cần trừ riêng.

## 19. Đáp án tham khảo cho bài tập

In [25]:
# Bài 1
r1 = (Solution().getSumAbsoluteDifferences([1, 1, 4]), Solution().getSumAbsoluteDifferences([3, 3, 3]))
print("Bài 1:", r1)
assert r1 == ([3, 3, 6], [0, 0, 0])

# Bài 2
r2 = trace_delta([1, 2, 4, 7, 11])
assert r2 == [20, 17, 15, 18, 30] == sum_abs_brute([1, 2, 4, 7, 11])

# Bài 3
def min_total_distance_sol(nums):
    n, total, left = len(nums), sum(nums), 0
    best, best_i = None, -1
    for i, x in enumerate(nums):
        v = total - 2 * left + x * (2 * i - n)
        if best is None or v < best:
            best, best_i = v, i
        left += x
    return best, best_i

rnd = random.Random(3)
for _ in range(1000):
    nums_ = sorted(rnd.randint(1, 50) for _ in range(rnd.randint(2, 20)))
    best, bi = min_total_distance_sol(nums_)
    res = sum_abs_brute(nums_)
    assert best == min(res) and res[bi] == best
print("Bài 3 OK | ví dụ 2:", min_total_distance_sol([1, 4, 6, 8, 10]))

Bài 1: ([3, 3, 6], [0, 0, 0])
  result[0] = total - n*nums[0] = 25 - 5*1 = 20
  i=1: d=1, 2i-n=-3, result[1] = 17
  i=2: d=2, 2i-n=-1, result[2] = 15
  i=3: d=3, 2i-n= 1, result[3] = 18
  i=4: d=4, 2i-n= 3, result[4] = 30
Bài 3 OK | ví dụ 2: (13, 2)


In [26]:
# Bài 4
def sum_abs_queries_sol(nums, queries):
    a = sorted(nums)
    pre = list(accumulate(a, initial=0))
    return [query_sum_abs(a, pre, q) for q in queries]

rnd = random.Random(4)
for _ in range(500):
    nums_ = [rnd.randint(1, 30) for _ in range(rnd.randint(1, 15))]
    qs = [rnd.randint(-5, 40) for _ in range(5)]
    assert sum_abs_queries_sol(nums_, qs) == [sum(abs(x - q) for x in nums_) for q in qs]
print("Bài 4 OK | [3,1,6,8], q=[1,5] ->", sum_abs_queries_sol([3, 1, 6, 8], [1, 5]))

# Bài 5
def sum_sq_diff_sol(nums):
    n = len(nums)
    s1 = sum(nums)
    s2 = sum(x * x for x in nums)
    return [n * x * x - 2 * x * s1 + s2 for x in nums]

rnd = random.Random(5)
for _ in range(500):
    nums_ = [rnd.randint(-20, 20) for _ in range(rnd.randint(1, 15))]
    assert sum_sq_diff_sol(nums_) == [sum((x - y) ** 2 for y in nums_) for x in nums_]
print("Bài 5 OK | [2,3,5] ->", sum_sq_diff_sol([2, 3, 5]))

Bài 4 OK | [3,1,6,8], q=[1,5] -> [14, 10]
Bài 5 OK | [2,3,5] -> [10, 5, 13]


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "tổng khoảng cách tới mọi phần tử" trên mảng đã sắp xếp:**

```python
n, total, left = len(a), sum(a), 0
res = []
for i, x in enumerate(a):
    right = total - left - x
    res.append(x * i - left + right - x * (n - 1 - i))   # = total - 2*left + x*(2i - n)
    left += x                                           # cập nhật SAU
return res
```

**Template sai phân:** `res[0] = total − n·a[0]`, `res[i] = res[i−1] + (a[i] − a[i−1])·(2i − n)`.

**Template truy vấn điểm `q`:** `k = bisect_left(a, q)`; `q·k − pre[k] + (pre[n] − pre[k]) − q·(n − k)`.

**Checklist:**
- [ ] Mảng đã sắp xếp (đề bảo đảm) — nếu không thì phải sắp kèm chỉ số?
- [ ] Số phần tử bên phải là `n − 1 − i` khi tổng bên phải không gồm `x`?
- [ ] `left` cập nhật **sau** khi dùng?
- [ ] Sai phân dùng `(2i − n)`, không đảo dấu?
- [ ] Kiểm với mảng toàn phần tử bằng nhau (toàn 0) và hai phần tử?
- [ ] Đã so với `O(n²)` trên mảng ngẫu nhiên đã sắp xếp?
- [ ] Chữ ký đúng: `def getSumAbsoluteDifferences(self, nums: List[int]) -> List[int]`?